In [2]:
import os
import importlib
# os.environ["CUDA_VISIBLE_DEVICES"]="2,3"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, AutoModel
from datasets import load_dataset
import torch
#from sentence_transformers import SentenceTransformer, InputExample, losses
#from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from collections import defaultdict
import re
import numpy as np
from tqdm import tqdm

/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data_dir = '/raid/deallab/SF_RAG_Data/ASQA'

qa_df=pd.read_csv(f'{data_dir}/test/qa_test.csv') #data=df[['question','long_answers']] # questions=data['question'] #references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]
qa_df.head()


,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,f743f676-48f8-42c6-ab8e-e4cd0a0542ce,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,40f108d2-081b-444a-b905-21dc7513628b,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,71925ca1-fcc5-4856-94cb-da3036123ca0,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,f9c40e99-b832-4f71-9d41-f821896e288c,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,2031b0f8-f0a1-4d5d-bed6-f706d1511d42,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [15]:
context=["Simon & Garfunkel, specifically Paul Simon, is the original artist of the song 'The Sound of Silence'.The original artist of 'The Sound of Silence' is the American folk rock duo Simon & Garfunkel, written by Paul Simon.The song 'The Sound of Silence' is a classic American folk rock song written by Paul Simon and performed by Simon & Garfunkel. It was originally recorded in an acoustic version in March 1964, but its electric remix, released in September 1965, became a huge success, reaching number one on the Billboard Hot 100 chart for two weeks. The song's themes of alienation, disconnection, and the search for meaning reflect the experiences of Paul Simon during his time in London, and its success led to the duo's reunion and the recording of their second album, 'Sounds of Silence', which is considered a masterpiece of folk rock.||"]

In [16]:
i=1
context=context[0]
question=qa_df['question'][i]
follow=eval(qa_df['follow_up_questions'][i])
short=eval(qa_df['short_answers'][i])

print(question)
print(follow)
print(short)

Who is the original artist of sound of silence?
['Who is the original artist of sound of silence, the song, released in 1964?', 'Who is the original artist of sound of silence, the album?', 'Who is the original artist of sound of silence, the song, released in 2016?']
[['Simon & Garfunkel', 'Paul Simon and Art Garfunkel', 'Art Garfunkel', 'Paul Simon'], ['Simon & Garfunkel', 'Paul Simon and Art Garfunkel', 'Art Garfunkel', 'Paul Simon'], ['Dami Im']]


In [6]:
short[0]

['Simon & Garfunkel',
 'Paul Simon and Art Garfunkel',
 'Art Garfunkel',
 'Paul Simon']

In [7]:
context[0]

'A'

In [8]:
follow[0]

'Who is the original artist of sound of silence, the song, released in 1964?'

In [9]:
import argparse
import collections
import json
import re
import string


def normalize_answer(s):
  """Lower text and remove punctuation, articles and extra whitespace."""

  def remove_articles(text):
    return re.sub(r'\b(a|an|the)\b', ' ', text)

  def white_space_fix(text):
    return ' '.join(text.split())

  def remove_punc(text):
    exclude = set(string.punctuation)
    return ''.join(ch for ch in text if ch not in exclude)

  def lower(text):
    return text.lower()

  return white_space_fix(remove_articles(remove_punc(lower(s))))

def _get_tokens(s):
  """Split the string into tokens.

  Args:
    s: string to be split

  Returns:
    list of tokens
  """

  if not s:
    return []
  return normalize_answer(s).split()

def _compute_f1(a_gold, a_pred):
  """Compute F1 score between two strings.

  Args:
    a_gold: string one
    a_pred: string two

  Returns:
        f1 score
  """

  gold_toks = _get_tokens(a_gold)
  pred_toks = _get_tokens(a_pred)

  common = collections.Counter(gold_toks) & collections.Counter(pred_toks)
  num_same = sum(common.values())

  if len(gold_toks) == 0 or len(pred_toks) == 0:
    # If either is no-answer, then F1 is 1 if they agree, 0 otherwise
    return int(gold_toks == pred_toks)

  if num_same == 0:
    return 0

  precision = 1.0 * num_same / len(pred_toks)
  recall = 1.0 * num_same / len(gold_toks)
  f1 = (2 * precision * recall) / (precision + recall)

  return f1

In [ ]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_name = "deepset/roberta-base-squad2"

prediction=[]
print(context)
print(follow)
print(short)
nlp = pipeline('question-answering', model=model_name, tokenizer=model_name, device=device)
cnt=0
loc_f1=0
print(len(follow))
for i in range(len(follow)):
    QA_input = {
        'question': follow[i],
        'context': context
    }
    print(follow[i])
    res = nlp(QA_input)
    prediction=[]
    prediction.append(res['answer'])
    print(res)
    print(len(short))
    ans=[]
    for a in short[i]:
        for p in prediction:
            res=_compute_f1(a, p)
            ans.append(res)
    loc_f1+=max(ans)
    cnt+=1
f1=loc_f1/cnt
print(f1)

Simon & Garfunkel, specifically Paul Simon, is the original artist of the song 'The Sound of Silence'.The original artist of 'The Sound of Silence' is the American folk rock duo Simon & Garfunkel, written by Paul Simon.The song 'The Sound of Silence' is a classic American folk rock song written by Paul Simon and performed by Simon & Garfunkel. It was originally recorded in an acoustic version in March 1964, but its electric remix, released in September 1965, became a huge success, reaching number one on the Billboard Hot 100 chart for two weeks. The song's themes of alienation, disconnection, and the search for meaning reflect the experiences of Paul Simon during his time in London, and its success led to the duo's reunion and the recording of their second album, 'Sounds of Silence', which is considered a masterpiece of folk rock.||
['Who is the original artist of sound of silence, the song, released in 1964?', 'Who is the original artist of sound of silence, the album?', 'Who is the o

In [11]:
prediction

['.||']

In [12]:
cnt=0
loc_f1=0
for answers in short:
    ans=[]
    for a in answers:
        for p in prediction:
            res=_compute_f1(a, p)
            ans.append(res)
    cnt+=1
    loc_f1+=max(ans)
f1=loc_f1/cnt

In [13]:
f1

0.0